# Distributed Memory Parallelization - MPI Hello World

So far, our application is able to utilize all available GPUs in a **single compute node**.
The next step is building on its concepts and leverage MPI to create an implementation capable of scaling across **multiple compute nodes**.

We start with a short introductory presentation that explains MPI through the lens of a CUDA programmer.

Next we look at a hello world example which has been extended to demonstrate additional functionalities.
The source code is available at [hello-world.cpp](../src/10-mpi-hw/hello-world.cpp).

To build it, we use the `mpic++` compiler wrapper.
Executing the next cell gives us additional information about the compiler wrapped.

In [ ]:
!mpic++ --version

Building with this wrapper works very much as it would with the wrapped compiler.

In [ ]:
!mpic++ -O3 -o ../build/10-mpi-hw-cpu ../src/10-mpi-hw/hello-world.cpp

Execution is then done by either
* simply executing the binary as usual, which spawns a single rank, or
* using the `mpirun` launcher which takes the `-n` parameter specifying the number of ranks to be launched.

In [ ]:
!mpirun -n 2 ../build/10-mpi-hw-cpu

## Aside: CUDA-Aware MPI

By default, MPI can only handle **host** data.
As consequence, communicating **device** data is usually a multi-stage process:
* The sender copies data from its device memory to its host memory
  * (and a temporary allocation must be made if none exists already)
* Data is communicated via MPI
* The receiver copies the received data from its host memory to its device memory

CUDA-aware MPI is able to handle device pointers as well.
Using the concept of UVA (see [08-p2p](./08-p2p.ipynb)), it can detect whether a pointer passed refers to host or device memory.

Note that this only guarantees, that the MPI implementation is able to work with device data.
It does *not* guarantee any optimizations.
In particular whether
* GPU Direct RDMA (see below) is used,
* potentially required transfers between host and device are asynchronous,
* potentially required transfers between host and device are done in streams,
* staging buffers are used (allocated) in any part of the communication.

It also gives no guarantees about how **intra-node** transfers are handled, although these are usually mapped to P2P transfers if possible (see [08-p2p](./08-p2p.ipynb)).

## Aside: GPU Direct RDMA

CUDA-aware MPI lays the foundation for GPU Direct RDMA which allows the third-party devices, such as an InfiniBand NIC, and a GPU memory to directly exchange data.
This, in essence, bypasses the host CPU and host memory for GPU-to-GPU communication across nodes.

As before, it requires compatible hardware (Kepler-generation or newer, and RDMA-capable NICs) and software/ driver stack (e.g. nvidia-peermem).

Without GPU Direct RDMA, transfers need to be relayed via the host.

<div style="text-align: center;">
  <img style="margin: 0 auto; height: 320px; background-color: white" src="img/gpu-direct-without.png" alt="Data Routing Visualization"/>
</div>

With GPU Direct RDMA, transfers are optimized.

<div style="text-align: center;">
  <img style="margin: 0 auto; height: 320px; background-color: white" src="img/gpu-direct-with.png" alt="Data Routing Visualization"/>
</div>

## CUDA-Aware MPI Hello World

Next we look at an updated version of our hello world example demonstrating CUDA interoperability.
The source code is available at [hello-world.cu](../src/10-mpi-hw/hello-world.cu).

To build it, we cannot use the `mpic++` compiler wrapper natively, at least not the way it is configured right now.
Instead, we use the nvcc and `mpic++` as downstream compiler.

In [ ]:
!nvcc -ccbin=mpic++ -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/10-mpi-hw-gpu ../src/10-mpi-hw/hello-world.cu

Execution is then done as before by either
* simply executing the binary as usual, which spawns a single rank, or
* using the `mpirun` launcher which takes the `-n` parameter specifying the number of ranks to be launched.

In [ ]:
!mpirun -n 2 ../build/10-mpi-hw-cpu

## Next Step

The next step is applying an MPI parallelization to our heat distribution simulation in [11](./11-mpi.ipynb).